In [97]:
import pandas as pd

# Assign the fully preprocessed dataset to df as requested
df = df_superstore.copy()

print("Data loaded successfully into 'df'!")
display(df.head())

Data loaded successfully into 'df'!


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Day of Week
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,2023,1,January,Tuesday
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,2023,1,January,Wednesday
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Labels,Avery 508,11.784,3,0.2,4.2717,2023,1,January,Wednesday
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,2023,1,January,Wednesday
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,2023,1,January,Thursday


In [98]:
# Print general info about the dataset
print(df_superstore.info())

# Count missing values in each column
missing_vals = df_superstore.isnull().sum()
print("\nMissing values per column:")
print(missing_vals[missing_vals > 0] if missing_vals.sum() > 0 else "No missing values found!")

# Check for duplicate rows
duplicates = df_superstore.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Row ID             10194 non-null  int64         
 1   Order ID           10194 non-null  object        
 2   Order Date         10194 non-null  datetime64[ns]
 3   Ship Date          10194 non-null  datetime64[ns]
 4   Ship Mode          10194 non-null  object        
 5   Customer ID        10194 non-null  object        
 6   Customer Name      10194 non-null  object        
 7   Segment            10194 non-null  object        
 8   Country/Region     10194 non-null  object        
 9   City               10194 non-null  object        
 10  State/Province     10194 non-null  object        
 11  Postal Code        10194 non-null  object        
 12  Region             10194 non-null  object        
 13  Product ID         10194 non-null  object        
 14  Catego

In [99]:
# Extract temporal features
df_superstore['Order Year'] = df_superstore['Order Date'].dt.year
df_superstore['Order Month'] = df_superstore['Order Date'].dt.month
df_superstore['Order Month Name'] = df_superstore['Order Date'].dt.strftime('%B')
df_superstore['Order Day of Week'] = df_superstore['Order Date'].dt.day_name()

# Display the updated DataFrame with new features
display(df_superstore[['Order Date', 'Order Year', 'Order Month', 'Order Month Name', 'Order Day of Week']].head())

,Order Date,Order Year,Order Month,Order Month Name,Order Day of Week
0,2023-01-03,2023,1,January,Tuesday
1,2023-01-04,2023,1,January,Wednesday
2,2023-01-04,2023,1,January,Wednesday
3,2023-01-04,2023,1,January,Wednesday
4,2023-01-05,2023,1,January,Thursday


In [100]:
# Show summary statistics of numerical fields
display(df_superstore[['Sales', 'Quantity', 'Discount', 'Profit']].describe())

# Check for any logical data anomalies
negative_sales = (df_superstore['Sales'] < 0).sum()
negative_quantity = (df_superstore['Quantity'] <= 0).sum()
invalid_discount = ((df_superstore['Discount'] < 0) | (df_superstore['Discount'] > 1)).sum()

print(f"Anomalies found:")
print(f"- Negative Sales: {negative_sales}")
print(f"- Non-positive Quantities: {negative_quantity}")
print(f"- Invalid Discounts (out of 0-1 range): {invalid_discount}")

,Sales,Quantity,Discount,Profit
count,10194.000000,10194.000000,10194.000000,10194.000000
mean,228.225854,3.791838,0.155385,28.673417
std,619.906839,2.228317,0.206249,232.465115
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.220000,2.000000,0.000000,1.760800
50%,53.910000,3.000000,0.200000,8.690000
75%,209.500000,5.000000,0.200000,29.297925
max,22638.480000,14.000000,0.800000,8399.976000


Anomalies found:
- Negative Sales: 0
- Non-positive Quantities: 0
- Invalid Discounts (out of 0-1 range): 0


In [101]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [102]:
# Initialize df_prep as a copy of df and encode key categorical columns
df_prep = df.copy()

le = LabelEncoder()
for col in ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']:
    df_prep[col] = le.fit_transform(df_prep[col].astype(str))

In [103]:
# Define our features (X) and target (y) for predicting Profit
feature_cols = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Order Year', 'Order Month']
X_superstore = df_prep[feature_cols]
y_superstore = df_prep['Profit']

In [104]:
# Split into training and test sets
X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(X_superstore, y_superstore, test_size=0.2, random_state=42)


In [105]:

df_prep = df_superstore.copy()

print("Preprocessed and split the dataset successfully:")
print(f"Training set size: {X_train_sp.shape[0]} samples")
print(f"Testing set size: {X_test_sp.shape[0]} samples")
display(X_train_sp.head())

Preprocessed and split the dataset successfully:
Training set size: 8155 samples
Testing set size: 2039 samples


,Ship Mode,Segment,Region,Category,Sub-Category,Sales,Quantity,Discount,Order Year,Order Month
5128,2,0,2,1,2,35.97,3,0.0,2025,6
8160,3,1,0,0,5,182.94,3,0.0,2026,7
8695,0,2,1,1,14,70.98,1,0.0,2026,9
7337,2,0,0,1,14,94.20,6,0.0,2026,4
8504,3,1,1,1,10,18.90,6,0.0,2026,9


In [106]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [107]:
# Create a copy for engineered modeling
df_advanced = df_superstore.copy()

# Feature Engineering: interaction term of Sales and Discount
df_advanced['Sales_x_Discount'] = df_advanced['Sales'] * df_advanced['Discount']


In [108]:
# Categorical and numerical columns
cat_cols = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
num_cols = ['Sales', 'Quantity', 'Discount', 'Sales_x_Discount', 'Order Year', 'Order Month']

# One-Hot Encode categorical variables
df_encoded = pd.get_dummies(df_advanced[cat_cols + num_cols], columns=cat_cols, drop_first=True)


In [109]:
# Split features and target
X_adv = df_encoded
y_adv = df_advanced['Profit']


In [110]:
# Split into training and test sets
X_train_adv, X_test_adv, y_train_adv, y_test_adv = train_test_split(
    X_adv, y_adv, test_size=0.2, random_state=42
)

In [111]:
# Initialize and train our High-Performance Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_adv, y_train_adv)

# Make predictions
y_pred_adv = rf_model.predict(X_test_adv)


In [112]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Calculate metrics for the Optimized Random Forest model
mse_adv = mean_squared_error(y_test_adv, y_pred_adv)
rmse_adv = np.sqrt(mse_adv)
r2_adv = r2_score(y_test_adv, y_pred_adv)

print("Optimized Random Forest Regressor Performance:")
print(f"Mean Squared Error (MSE): {mse_adv:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_adv:.4f}")
print(f"R-squared (R2) Score: {r2_adv:.4f}")

Optimized Random Forest Regressor Performance:
Mean Squared Error (MSE): 19647.1605
Root Mean Squared Error (RMSE): 140.1683
R-squared (R2) Score: 0.7810


In [113]:


# Create a copy for the high-accuracy Linear Regression pipeline
df_lr = df_superstore.copy()

# Cap extreme values to reduce outlier distortion
sales_cap = df_lr['Sales'].quantile(0.99)
profit_cap_low = df_lr['Profit'].quantile(0.01)
profit_cap_high = df_lr['Profit'].quantile(0.99)

df_lr['Sales'] = np.clip(df_lr['Sales'], 0, sales_cap)
df_lr['Profit'] = np.clip(df_lr['Profit'], profit_cap_low, profit_cap_high)

# Apply log transform to Sales (add 1 to prevent log(0))
df_lr['Log_Sales'] = np.log1p(df_lr['Sales'])

print("Outliers handled and Sales log-transformed successfully!")

Outliers handled and Sales log-transformed successfully!


In [114]:
# Generate explicit interaction terms for the linear model
df_lr['Sales_x_Discount'] = df_lr['Sales'] * df_lr['Discount']
df_lr['Log_Sales_x_Discount'] = df_lr['Log_Sales'] * df_lr['Discount']
df_lr['Sales_Sq'] = df_lr['Sales'] ** 2

print("Interaction and polynomial features engineered.")

Interaction and polynomial features engineered.


In [115]:
# Define categorical and numerical features
cat_cols_lr = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
num_cols_lr = ['Sales', 'Log_Sales', 'Quantity', 'Discount', 'Sales_x_Discount', 'Log_Sales_x_Discount', 'Sales_Sq', 'Order Year', 'Order Month']

# One-hot encode the categorical fields
df_lr_encoded = pd.get_dummies(df_lr[cat_cols_lr + num_cols_lr], columns=cat_cols_lr, drop_first=True)

# Define features (X) and target (y)
X_lr = df_lr_encoded
y_lr = df_lr['Profit']

print(f"One-hot encoding complete. Features prepared: {X_lr.shape[1]}")

One-hot encoding complete. Features prepared: 35


In [116]:
from sklearn.linear_model import LinearRegression

# Train-test split
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

# Train optimized Linear Regression
lr_model_opt = LinearRegression()
lr_model_opt.fit(X_train_lr, y_train_lr)

LinearRegression()

In [117]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Predict and score
y_pred_lr = lr_model_opt.predict(X_test_lr)

mse_lr = mean_squared_error(y_test_lr, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test_lr, y_pred_lr)

print("Optimized Linear Regression Performance:")
print(f"Mean Squared Error (MSE): {mse_lr:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lr:.4f}")
print(f"R-squared (R2) Score: {r2_lr:.4f}")

Optimized Linear Regression Performance:
Mean Squared Error (MSE): 2335.7176
Root Mean Squared Error (RMSE): 48.3293
R-squared (R2) Score: 0.7753


In [118]:


# Create a comparison DataFrame for 10 samples
comparison_df = pd.DataFrame({
    'Actual Profit': y_test_lr,
    'Predicted Profit': y_pred_lr
}).head(10)

# Calculate the difference/error
comparison_df['Difference (Actual - Predicted)'] = comparison_df['Actual Profit'] - comparison_df['Predicted Profit']

# Display the table formatted to 2 decimal places
display(comparison_df.round(2))

,Actual Profit,Predicted Profit,Difference (Actual - Predicted)
6636,-14.33,-20.62,6.29
3742,4.77,-4.98,9.75
5114,-7.71,-31.63,23.92
6505,108.18,169.83,-61.65
10072,218.25,214.30,3.96
6520,570.73,432.09,138.64
3149,48.96,66.89,-17.92
2876,20.68,31.82,-11.14
5610,1.79,1.32,0.46
7742,28.00,53.16,-25.16


In [119]:
from sklearn.model_selection import train_test_split

# Prepare the features and target
X_dt = df_lr_encoded
y_dt = df_lr['Profit']

In [120]:
# Split into training and testing datasets (80% train, 20% test)
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_dt, y_dt, test_size=0.2, random_state=42
)

In [121]:
from sklearn.model_selection import train_test_split

# Define the features and target for the decision tree model
X_dt = df_lr_encoded
y_dt = df_lr['Profit']

# Split into training and testing datasets (80% train, 20% test)
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_dt, y_dt, test_size=0.2, random_state=42
)

print("Data prepared and split successfully:")
print(f"Training set shape: {X_train_dt.shape}")
print(f"Testing set shape: {X_test_dt.shape}")

Data prepared and split successfully:
Training set shape: (8155, 35)
Testing set shape: (2039, 35)


In [122]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV


In [123]:
# Define parameter grid to find the optimal balance of depth and leaf size
param_grid = {
    'max_depth': [4, 6, 8, 10, 12],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [124]:

# Define parameter grid to find the optimal balance of depth and leaf size
param_grid = {
    'max_depth': [4, 6, 8, 10, 12],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10]
}


In [125]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

# Define parameter grid to find the optimal balance of depth and leaf size
param_grid = {
    'max_depth': [4, 6, 8, 10, 12],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [126]:
# Fit the grid search to training data
grid_search.fit(X_train_dt, y_train_dt)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [4, 6, 8, 10, 12],
                         'min_samples_leaf': [2, 5, 10],
                         'min_samples_split': [5, 10, 20]},
             scoring='r2')

In [127]:
best_dt_model = grid_search.best_estimator_
print("Grid search complete!")
print(f"Best Hyperparameters: {grid_search.best_params_}")

Grid search complete!
Best Hyperparameters: {'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 10}


In [128]:
# Make predictions on the test set
y_pred_dt = best_dt_model.predict(X_test_dt)

# Calculate metrics
mse_dt = mean_squared_error(y_test_dt, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test_dt, y_pred_dt)

print("Optimized Decision Tree Performance:")
print(f"Mean Squared Error (MSE): {mse_dt:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_dt:.4f}")
print(f"R-squared (R2) Score: {r2_dt:.4f}")

Optimized Decision Tree Performance:
Mean Squared Error (MSE): 1323.5681
Root Mean Squared Error (RMSE): 36.3809
R-squared (R2) Score: 0.8727


In [129]:
# Create a comparison DataFrame for the Decision Tree predictions
dt_comparison = pd.DataFrame({
    'Actual Profit': y_test_dt,
    'Predicted Profit': y_pred_dt
}).head(10)

# Calculate the error difference
dt_comparison['Difference (Actual - Predicted)'] = dt_comparison['Actual Profit'] - dt_comparison['Predicted Profit']

# Display the table formatted to 2 decimal places
display(dt_comparison.round(2))

,Actual Profit,Predicted Profit,Difference (Actual - Predicted)
6636,-14.33,-13.88,-0.45
3742,4.77,4.48,0.29
5114,-7.71,-56.92,49.21
6505,108.18,136.75,-28.58
10072,218.25,288.08,-69.82
6520,570.73,351.27,219.46
3149,48.96,56.12,-7.16
2876,20.68,23.22,-2.54
5610,1.79,1.67,0.12
7742,28.00,23.22,4.78


In [130]:

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Ensure df_lr is defined. Since it wasn't initialized earlier, we'll create it from df
if 'df_lr' not in locals():
    df_lr = df.copy()

# Apply necessary feature transformations if they don't exist
if 'Log_Sales' not in df_lr.columns:
    df_lr['Log_Sales'] = np.log1p(df_lr['Sales'])
if 'Sales_x_Discount' not in df_lr.columns:
    df_lr['Sales_x_Discount'] = df_lr['Sales'] * df_lr['Discount']
if 'Log_Sales_x_Discount' not in df_lr.columns:
    df_lr['Log_Sales_x_Discount'] = df_lr['Log_Sales'] * df_lr['Discount']
if 'Sales_Sq' not in df_lr.columns:
    df_lr['Sales_Sq'] = df_lr['Sales'] ** 2

cat_cols_lr = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
num_cols_lr = ['Sales', 'Log_Sales', 'Quantity', 'Discount', 'Sales_x_Discount', 'Log_Sales_x_Discount', 'Sales_Sq', 'Order Year', 'Order Month']
df_lr_encoded = pd.get_dummies(df_lr[cat_cols_lr + num_cols_lr], columns=cat_cols_lr, drop_first=True)

X_svm = df_lr_encoded
y_svm = df_lr['Profit']

# Split into training and test sets
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_svm_scaled = scaler.fit_transform(X_train_svm)
X_test_svm_scaled = scaler.transform(X_test_svm)

print("Features scaled successfully for Support Vector Regression!")

Features scaled successfully for Support Vector Regression!


In [131]:
from sklearn.svm import SVR

# Initialize SVR with standard hyper-parameters
svr_model = SVR(kernel='rbf', C=100.0, epsilon=0.1)

# Train the model
svr_model.fit(X_train_svm_scaled, y_train_svm)

# Make predictions on test set
y_pred_svm = svr_model.predict(X_test_svm_scaled)
print("SVR model trained and predictions generated!")

SVR model trained and predictions generated!


### Step 3: Evaluate SVR Performance
Let's calculate the performance metrics (MSE, RMSE, R-squared) for the SVR model and compare its predictions with actual values.

In [132]:
# Calculate performance metrics
mse_svm = mean_squared_error(y_test_svm, y_pred_svm)
rmse_svm = np.sqrt(mse_svm)
r2_svm = r2_score(y_test_svm, y_pred_svm)

print("Support Vector Regression Performance:")
print(f"Mean Squared Error (MSE): {mse_svm:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_svm:.4f}")
print(f"R-squared (R2) Score: {r2_svm:.4f}")

Support Vector Regression Performance:
Mean Squared Error (MSE): 1293.6191
Root Mean Squared Error (RMSE): 35.9669
R-squared (R2) Score: 0.8756


In [133]:


# Create a comparison DataFrame for the SVM/SVR predictions
svm_comparison = pd.DataFrame({
    'Actual Profit': y_test_svm,
    'Predicted Profit': y_pred_svm
}).head(10)

# Calculate the error difference
svm_comparison['Difference (Actual - Predicted)'] = svm_comparison['Actual Profit'] - svm_comparison['Predicted Profit']

# Display the table formatted to 2 decimal places
display(svm_comparison.round(2))

,Actual Profit,Predicted Profit,Difference (Actual - Predicted)
6636,-14.33,-18.32,3.99
3742,4.77,2.99,1.79
5114,-7.71,-55.41,47.70
6505,108.18,117.74,-9.56
10072,218.25,230.65,-12.40
6520,570.73,488.42,82.31
3149,48.96,51.20,-2.24
2876,20.68,18.75,1.93
5610,1.79,1.01,0.78
7742,28.00,27.32,0.68


In [135]:
import pickle

# Save the SVR model
with open('svr_model.pkl', 'wb') as model_file:
    pickle.dump(svr_model, model_file)

# Save the scaler
with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

print("Model 'svr_model.pkl' and Scaler 'scaler.pkl' saved successfully!")

Model 'svr_model.pkl' and Scaler 'scaler.pkl' saved successfully!
